# RAPID2 Tutorial on MAAP DPS

This notebook recreates the [RAPID2 Tutorial](https://github.com/c-h-david/rapid2/blob/main/TUTORIAL.md) on the MAAP platform.

**Architecture:**
- **Notebook (local):** Lightweight pre-processing — download runoff data, transform to external inflow, generate cold start
- **DPS Job (remote):** Compute-heavy Muskingum routing via `rapid2`

**Prerequisites:**
- Algorithm `rapid2` (version `maap`) is already registered on MAAP DPS
- Earthdata credentials are configured (auto-provided on MAAP)
- Basin: `pfaf_74` (Mississippi River Basin)

## Setup

In [ ]:
!pip install -q rapid2

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path
from maap.maap import MAAP

maap = MAAP(maap_host="api.maap-project.org")

BASIN = "pfaf_74"
WORK_DIR = Path("rapid2_tutorial")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S3_PREFIX = f"s3://my-private-bucket/rapid2/tutorial/{BASIN}/"

## Step 1: Download Static Files from Zenodo

Download connectivity, coordinates, and coupling files for `pfaf_74` from Zenodo DOI [10.5281/zenodo.8248069](https://doi.org/10.5281/zenodo.8248069).

In [ ]:
ZENODO_RECORD = "10013744"
ZENODO_BASE = f"https://zenodo.org/records/{ZENODO_RECORD}/files"

# Zenodo uses "pfaf_ii" as a template name; ZIPs contain per-basin files inside
ZIP_FILES = [
    "rapid_connect_pfaf_ii.zip",
    "coords_pfaf_ii.zip",
    "rapid_coupling_pfaf_ii_GLDAS.zip",
    "k_pfaf_ii_nrm.zip",
    "x_pfaf_ii_nrm.zip",
    "riv_bas_id_pfaf_ii_topo.zip",
]

for zf in ZIP_FILES:
    url = f"{ZENODO_BASE}/{zf}?download=1"
    dest = INPUT_DIR / zf
    if not dest.exists():
        print(f"Downloading {zf}...")
        urllib.request.urlretrieve(url, dest)
    with zipfile.ZipFile(dest, "r") as z:
        z.extractall(INPUT_DIR)

print("Static files:")
for f in sorted(INPUT_DIR.glob(f"*{BASIN}*")):
    print(f"  {f.name}")

## Step 2: Download GLDAS Runoff Data

Uses `dgldas2` to download GLDAS phase 2.1, VIC model, for January 2010.

In [ ]:
GLDAS_FILE = INPUT_DIR / f"GLDAS_2.1_VIC_2010-01.nc4"

!dgldas2 \
    --phase 2.1 \
    --model VIC \
    --time 2010-01 \
    --land_surface_model \
    {GLDAS_FILE}

## Step 3: Transform Runoff to External Inflow

Uses `cpllsm` to couple the land surface model output with the river network.

In [ ]:
QEXT_FILE = INPUT_DIR / f"Qext_{BASIN}_GLDAS_2.1_VIC_2010-01.nc4"

!cpllsm \
    --land_surface_model \
    {GLDAS_FILE} \
    --connectivity \
    {INPUT_DIR}/rapid_connect_{BASIN}.csv \
    --coordinates \
    {INPUT_DIR}/coords_{BASIN}.csv \
    --coupling \
    {INPUT_DIR}/rapid_coupling_{BASIN}_GLDAS.csv \
    --external_inflow \
    {QEXT_FILE}

## Step 4: Generate Cold Start (Zero Initial Discharge)

Uses `zeroqinit` to create an initial condition file with all flows set to zero.

In [ ]:
QINIT_FILE = INPUT_DIR / f"Qinit_{BASIN}_GLDAS_2.1_VIC_2010-01.nc4"

!zeroqinit \
    --external_inflow \
    {QEXT_FILE} \
    --initial_outflow \
    {QINIT_FILE}

## Step 5: Upload Inputs to MAAP S3

Upload the 6 input files needed by the DPS job to your MAAP private bucket.

In [ ]:
DPS_INPUTS = {
    "Qex_ncf": QEXT_FILE,
    "Q00_ncf": QINIT_FILE,
    "con_csv": INPUT_DIR / f"rapid_connect_{BASIN}.csv",
    "kpr_csv": INPUT_DIR / f"k_{BASIN}_nrm.csv",
    "xpr_csv": INPUT_DIR / f"x_{BASIN}_nrm.csv",
    "bas_csv": INPUT_DIR / f"riv_bas_id_{BASIN}_topo.csv",
}

s3_urls = {}
for key, local_path in DPS_INPUTS.items():
    s3_path = f"{S3_PREFIX}{local_path.name}"
    !aws s3 cp {local_path} {s3_path}
    s3_urls[key] = s3_path
    print(f"  {key}: {s3_path}")

## Step 6: Submit DPS Job for Routing

Submit the compute-heavy `rapid2` Muskingum routing as a DPS job. This is the step that benefits from scaling — it iterates over every river reach at every time step.

In [ ]:
IS_dtR = "900"

job = maap.submitJob(
    identifier="rapid2_tutorial_pfaf_74",
    algo_id="rapid2",
    version="maap",
    queue="maap-dps-worker-8gb",
    username=maap.profile.account_info()["username"],
    Qex_ncf=s3_urls["Qex_ncf"],
    Q00_ncf=s3_urls["Q00_ncf"],
    con_csv=s3_urls["con_csv"],
    kpr_csv=s3_urls["kpr_csv"],
    xpr_csv=s3_urls["xpr_csv"],
    bas_csv=s3_urls["bas_csv"],
    IS_dtR=IS_dtR,
)

print(f"Job submitted: {job.id}")

In [ ]:
status = job.wait_for_completion()
print(f"Job status: {status}")

## Step 7: Download and Verify Outputs

Download the routing outputs from DPS and run `cmpncf` to compare against gold standard files (if available).

In [ ]:
for output_url in job.outputs:
    filename = output_url.split("/")[-1]
    local_out = OUTPUT_DIR / filename
    !aws s3 cp {output_url} {local_out}
    print(f"Downloaded: {local_out}")

In [ ]:
QOUT_FILE = OUTPUT_DIR / f"Qout_maap.nc4"

if QOUT_FILE.exists():
    print(f"Routing output: {QOUT_FILE}")
    print(f"Output size: {QOUT_FILE.stat().st_size / 1024:.1f} KB")
else:
    print("Output file not found — check DPS job logs.")

## Multi-Basin Scaling

To run RAPID2 across many basins, repeat Steps 1–5 for each basin and submit parallel DPS jobs. The cell below demonstrates this pattern.

In [ ]:
# Example: submit jobs for multiple basins in parallel
# Uncomment and customize the BASINS list for your use case.

# BASINS = ["pfaf_74", "pfaf_72", "pfaf_71"]
# jobs = {}
#
# for basin in BASINS:
#     s3_base = f"s3://my-private-bucket/rapid2/tutorial/{basin}/"
#     job = maap.submitJob(
#         identifier=f"rapid2_{basin}",
#         algo_id="rapid2",
#         version="maap",
#         queue="maap-dps-worker-8gb",
#         username=maap.profile.account_info()["username"],
#         Qex_ncf=f"{s3_base}Qext_{basin}_GLDAS_2.1_VIC_2010-01.nc4",
#         Q00_ncf=f"{s3_base}Qinit_{basin}_GLDAS_2.1_VIC_2010-01.nc4",
#         con_csv=f"{s3_base}rapid_connect_{basin}.csv",
#         kpr_csv=f"{s3_base}k_{basin}_nrm.csv",
#         xpr_csv=f"{s3_base}x_{basin}_nrm.csv",
#         bas_csv=f"{s3_base}riv_bas_id_{basin}_topo.csv",
#         IS_dtR="900",
#     )
#     jobs[basin] = job
#     print(f"Submitted {basin}: {job.id}")
#
# for basin, job in jobs.items():
#     status = job.wait_for_completion()
#     print(f"{basin}: {status}")